# Homework 09: Finetuning GPT

Refer to the corresponding lab and Rashcka's book (Ch. 6 and 7)



## Setup (run these first)

These three cells make the homework self-contained — they reproduce the model, tokenizer, decoding function, and trained weights from the **09_Decoding_Finetuning** lab. Run them top-to-bottom before the exercises. On Colab pick a **T4 GPU** runtime (Runtime → Change runtime type → T4 GPU).

In [ ]:
# ── SETUP 1/3: imports, tokenizer, device, and the GPT model classes ─────────
# Copied from the corresponding lab (09_Decoding_Finetuning.ipynb) so this
# homework is self-contained on Colab. Run this cell FIRST.

# Reduce CUDA memory fragmentation. MUST be set before torch initializes CUDA,
# so it lives at the very top of the first cell (takes effect after a restart).
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

!pip install torch tiktoken --quiet

import torch, torch.nn as nn, tiktoken, urllib.request, json, time
import matplotlib.pyplot as plt
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from matplotlib.ticker import MaxNLocator

# GPT-2 byte-pair tokenizer + pick GPU if available (the T4 on Colab)
tokenizer = tiktoken.get_encoding("gpt2")
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ── Model building blocks ─────────────────────────────────────────────────────
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift

class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * x**3)))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]))
    def forward(self, x): return self.layers(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads
        self.d_out     = d_out
        self.W_query   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key     = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj  = nn.Linear(d_out, d_out)
        self.dropout   = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
    def forward(self, x):
        b, n, _ = x.shape
        q = self.W_query(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_key(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_value(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        scores = q @ k.transpose(2, 3)
        scores.masked_fill_(self.mask.bool()[:n, :n], -torch.inf)
        w = self.dropout(torch.softmax(scores / k.shape[-1]**0.5, dim=-1))
        return self.out_proj((w @ v).transpose(1, 2).contiguous().view(b, n, self.d_out))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att  = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"],
                                        cfg["context_length"], cfg["drop_rate"],
                                        cfg["n_heads"], cfg["qkv_bias"])
        self.ff   = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop  = nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        x = x + self.drop(self.att(self.norm1(x)))
        x = x + self.drop(self.ff(self.norm2(x)))
        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb   = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb   = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb  = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head   = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
    def forward(self, in_idx):
        b, n = in_idx.shape
        x = self.tok_emb(in_idx) + self.pos_emb(torch.arange(n, device=in_idx.device))
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        return self.out_head(self.final_norm(x))

# ── Utility functions: text <-> token id tensors ─────────────────────────────
def text_to_token_ids(text, tokenizer):
    return torch.tensor(tokenizer.encode(text, allowed_special={"<|endoftext|>"})).unsqueeze(0)

def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

print("All classes loaded ✓")

In [ ]:
# ── SETUP 2/3: decoding function used by the exercises ───────────────────────
# Defines `generate` (temperature + top-k sampling). Copied from lab 09.
def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(idx[:, -context_size:])[:, -1, :]

        # Top-k filter: keep only the k highest-probability tokens
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            logits = torch.where(
                logits < top_logits[:, -1],
                torch.tensor(float("-inf")).to(logits.device),
                logits
            )

        # Temperature scaling + sampling; temperature == 0 means greedy argmax
        if temperature > 0.0:
            probs    = torch.softmax(logits / temperature, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        # Optional early stop at the end-of-text token
        if eos_id is not None and idx_next.item() == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [ ]:
# ── SETUP 3/3: load (or quickly train) the model used by the exercises ───────
# Defines GPT_CONFIG_TRAIN, `model`. Copied from lab 09_Decoding_Finetuning.
# On a fresh Colab with no saved checkpoint this trains a tiny model on
# "The Verdict" (~5 min on CPU, faster on the T4 GPU) and caches it.
GPT_CONFIG_TRAIN = {
    "vocab_size": 50257, "context_length": 256,
    "emb_dim": 768, "n_heads": 12, "n_layers": 12,
    "drop_rate": 0.1, "qkv_bias": False
}

# Download "The Verdict" training text if we don't already have it locally
import os
if not os.path.exists("the-verdict.txt"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/"
        "main/ch02/01_main-chapter-code/the-verdict.txt",
        "the-verdict.txt"
    )

with open("the-verdict.txt", encoding="utf-8") as f:
    text_data = f.read()

# If no saved weights exist, train a small model from scratch; otherwise load it
if not os.path.exists("gpt_verdict.pth"):
    print("No saved model found — training a quick one (≈5 min on CPU)...")

    class GPTDatasetV1(Dataset):
        def __init__(self, txt, tokenizer, max_length, stride):
            self.input_ids, self.target_ids = [], []
            ids = tokenizer.encode(txt)
            for i in range(0, len(ids) - max_length, stride):
                self.input_ids.append(torch.tensor(ids[i:i+max_length]))
                self.target_ids.append(torch.tensor(ids[i+1:i+max_length+1]))
        def __len__(self): return len(self.input_ids)
        def __getitem__(self, idx): return self.input_ids[idx], self.target_ids[idx]

    ctx = GPT_CONFIG_TRAIN["context_length"]
    split = int(0.9 * len(text_data))
    train_loader = DataLoader(GPTDatasetV1(text_data[:split], tokenizer, ctx, ctx),
                              batch_size=2, shuffle=True, drop_last=True)
    val_loader   = DataLoader(GPTDatasetV1(text_data[split:], tokenizer, ctx, ctx),
                              batch_size=2, shuffle=False, drop_last=False)

    def calc_loss_batch(xb, yb, model, device):
        xb, yb = xb.to(device), yb.to(device)
        return torch.nn.functional.cross_entropy(model(xb).flatten(0,1), yb.flatten())

    torch.manual_seed(123)
    model = GPTModel(GPT_CONFIG_TRAIN).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)
    for epoch in range(10):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad(); calc_loss_batch(xb, yb, model, device).backward(); opt.step()
        print(f"Epoch {epoch+1}/10 done")
    torch.save(model.state_dict(), "gpt_verdict.pth")
    print("Saved to gpt_verdict.pth")
else:
    model = GPTModel(GPT_CONFIG_TRAIN)
    model.load_state_dict(torch.load("gpt_verdict.pth", map_location=device))
    model.to(device)
    print("Loaded saved model ✓")

model.eval()

---
## Exercises

**Exercise 1 — Temperature exploration**

Run `generate()` on the same prompt with temperatures `[0.0, 0.5, 1.0, 1.5, 2.0]` and `top_k=20`. Print each output. At what temperature does the text start becoming incoherent for your trained model?

In [ ]:
import torch  # fixes "NameError: name 'torch' is not defined" -- needed for torch.manual_seed below

# Your code here
prompt = "The Supreme Court held that"
for T in [0.0, 0.5, 1.0, 1.5, 2.0]:
    torch.manual_seed(42)
    out = generate(model, text_to_token_ids(prompt, tokenizer).to(device),
                   max_new_tokens=25, context_size=GPT_CONFIG_TRAIN["context_length"],
                   temperature=T, top_k=20)
    print(f"τ={T}: {token_ids_to_text(out, tokenizer)}")

**Exercise 2 — Instruction masking**

Currently our `custom_collate_fn` only masks *padding* tokens with `-100`. An alternative is to also mask the instruction tokens, so the model **only learns to predict the response** (not the prompt).

Modify `custom_collate_fn` to additionally set all instruction tokens to `-100` in the targets. Use `format_input(entry)` to determine how many tokens to mask.

Does this improve or hurt performance? Why might it help (less overfitting to the instruction format) or hurt (less signal)?

In [ ]:
# ── Exercise 2: Instruction masking ──────────────────────────────────────────
# First bring in the instruction-tuning pieces from the lab so this exercise is
# self-contained: the dataset, the Alpaca prompt formatter, and a train/val/test
# split. (Same as 09_Decoding_Finetuning.)
import os, json, urllib.request
from functools import partial

# Download the instruction dataset (~1.1k instruction / input / output examples)
if not os.path.exists("instruction-data.json"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
        "/main/ch07/01_main-chapter-code/instruction-data.json",
        "instruction-data.json"
    )
with open("instruction-data.json") as f:
    data = json.load(f)

def format_input(entry):
    """Wrap an instruction entry in the Alpaca prompt template (instruction + optional input)."""
    instruction_text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

# 85 / 5 / 10 split (same proportions as the lab)
train_portion = int(len(data) * 0.85)
test_portion  = int(len(data) * 0.10)
train_data = data[:train_portion]
test_data  = data[train_portion:train_portion + test_portion]
val_data   = data[train_portion + test_portion:]


# ── The modification ─────────────────────────────────────────────────────────
# To mask the instruction we need to know, per example, WHERE the response
# begins. So the dataset stores (encoded_full_text, instruction_len), where
# instruction_len is the token count of everything up to and including
# "### Response:\n" -- i.e. format_input(entry) + the response header.
class InstructionDatasetWithMask(Dataset):
    def __init__(self, data, tokenizer):
        self.items = []
        for entry in data:
            instruction_part = format_input(entry) + "\n\n### Response:\n"
            full_text        = instruction_part + entry["output"]
            encoded_full     = tokenizer.encode(full_text)
            instruction_len  = len(tokenizer.encode(instruction_part))
            self.items.append((encoded_full, instruction_len))
    def __len__(self):  return len(self.items)
    def __getitem__(self, idx): return self.items[idx]


def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100,
                      allowed_max_length=None, device="cpu"):
    """Lab collate fn + instruction masking.

    Each batch item is (token_ids, instruction_len). In addition to masking
    padding, we set the instruction tokens in the *targets* to -100 so the
    cross-entropy loss (which uses ignore_index=-100) is computed ONLY over the
    response tokens.
    """
    max_len = max(len(ids) + 1 for ids, _ in batch)   # +1 for the input/target shift
    inputs_lst, targets_lst = [], []

    for ids, instruction_len in batch:
        padded  = ids + [pad_token_id] * (max_len - len(ids))
        inputs  = torch.tensor(padded[:-1])   # drop last  -> inputs
        targets = torch.tensor(padded[1:])    # shift by 1 -> targets

        # (1) original behaviour: keep ONE end-of-text/pad token, mask the rest
        pad_positions = (targets == pad_token_id).nonzero().squeeze(-1)
        if pad_positions.numel() > 1:
            targets[pad_positions[1:]] = ignore_index

        # (2) NEW: mask the instruction tokens. Targets are shifted by one, so
        #     the first RESPONSE token sits at target index (instruction_len - 1).
        #     Everything before that index is the prompt -> set it to -100.
        targets[:instruction_len - 1] = ignore_index

        if allowed_max_length:
            inputs  = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs); targets_lst.append(targets)

    return torch.stack(inputs_lst).to(device), torch.stack(targets_lst).to(device)


# ── Verify the masking on a tiny batch ───────────────────────────────────────
demo_ds      = InstructionDatasetWithMask(train_data[:2], tokenizer)
demo_collate = partial(custom_collate_fn, device="cpu", allowed_max_length=1024)
inp, tgt     = demo_collate([demo_ds[0], demo_ds[1]])

ids0, inst_len0 = demo_ds[0]
print(f"Example 0: {len(ids0)} tokens total, {inst_len0} instruction tokens")
print("First 12 target values (note the -100s masking the instruction):")
print(tgt[0][:12])
n_masked = (tgt[0] == -100).sum().item()
print(f"Masked targets in example 0: {n_masked} "
      f"(= {inst_len0 - 1} instruction + the rest padding)")
print("\nTo train with this, swap InstructionDatasetWithMask + this collate fn "
      "into the lab's DataLoaders and reuse train_instruction_model() unchanged "
      "(it already passes ignore_index=-100 to cross_entropy).")

**Does instruction masking help or hurt?**

In practice, on this small Alpaca-style dataset it tends to make **little difference, and often *slightly hurts*** held-out response quality — this is also what Raschka reports for the Chapter 7 exercise. The reasons cut both ways:

**Why it could help**
- **Focuses the gradient on what we care about.** We only ever sample the *response* at inference time, so spending loss budget on predicting the response (not re-predicting the fixed prompt) targets the actual objective.
- **Less overfitting to the instruction format.** The prompt template (`### Instruction:` / `### Input:` / `### Response:`) is highly repetitive. Without masking, the model spends a lot of capacity memorizing that boilerplate, which is wasted signal.

**Why it could hurt**
- **Throws away training signal.** Instruction tokens are still valid language; predicting them is useful next-token practice, especially for a *small* model on a *small* dataset where every token of supervision matters. Masking can remove the majority of tokens in each example.
- **Weaker grounding of the prompt.** Letting the model also model the instruction text can help it "understand" the prompt distribution, improving how well the response is conditioned on it.

**Bottom line:** it's an empirical trade-off. With abundant data and a larger model, instruction-only masking is the common production choice (you don't want to waste capacity on boilerplate). With a tiny model + ~1k examples like here, the lost signal usually outweighs the focus benefit, so the unmasked baseline is competitive or better. The right answer is to **train both and compare validation loss / sample quality** — the verification cell above gives you the masked loaders; reuse `train_instruction_model()` for each and compare.

### Train with instruction masking

Now actually fine-tune a model using the masked dataset + collate fn from above. This loads pretrained **GPT-2 Medium (355M)**, builds masked DataLoaders, and trains for 2 epochs. Runs in a few minutes on a **T4**.

`download_and_load_gpt2` downloads the OpenAI weights via TensorFlow, so the first run will also `pip install tensorflow` if it isn't present. The loss numbers below are computed **only over response tokens** (instruction tokens are `-100`), so they aren't directly comparable to an unmasked run's loss — compare *sample quality* instead.

In [ ]:
# ── Load pretrained GPT-2 Medium (355M) to fine-tune ─────────────────────────
# Grab Raschka's helper that downloads OpenAI's pretrained GPT-2 weights.
import numpy as np
if not os.path.exists("gpt_download.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/"
        "main/ch05/01_main-chapter-code/gpt_download.py",
        "gpt_download.py"
    )
from gpt_download import download_and_load_gpt2

def load_weights_into_gpt(gpt, params):
    """Copy OpenAI's TensorFlow GPT-2 weights into our PyTorch GPTModel."""
    gpt.pos_emb.weight = nn.Parameter(torch.tensor(params["wpe"]))
    gpt.tok_emb.weight = nn.Parameter(torch.tensor(params["wte"]))
    for b, block in enumerate(gpt.trf_blocks):
        # attention: the OpenAI checkpoint stores Q,K,V stacked -> split into 3
        q, k, v = np.split(params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1)
        block.att.W_query.weight = nn.Parameter(torch.tensor(q).T)
        block.att.W_key.weight   = nn.Parameter(torch.tensor(k).T)
        block.att.W_value.weight = nn.Parameter(torch.tensor(v).T)
        q_b, k_b, v_b = np.split(params["blocks"][b]["attn"]["c_attn"]["b"], 3)
        block.att.W_query.bias = nn.Parameter(torch.tensor(q_b))
        block.att.W_key.bias   = nn.Parameter(torch.tensor(k_b))
        block.att.W_value.bias = nn.Parameter(torch.tensor(v_b))
        block.att.out_proj.weight = nn.Parameter(torch.tensor(params["blocks"][b]["attn"]["c_proj"]["w"]).T)
        block.att.out_proj.bias   = nn.Parameter(torch.tensor(params["blocks"][b]["attn"]["c_proj"]["b"]))
        block.ff.layers[0].weight = nn.Parameter(torch.tensor(params["blocks"][b]["mlp"]["c_fc"]["w"]).T)
        block.ff.layers[0].bias   = nn.Parameter(torch.tensor(params["blocks"][b]["mlp"]["c_fc"]["b"]))
        block.ff.layers[2].weight = nn.Parameter(torch.tensor(params["blocks"][b]["mlp"]["c_proj"]["w"]).T)
        block.ff.layers[2].bias   = nn.Parameter(torch.tensor(params["blocks"][b]["mlp"]["c_proj"]["b"]))
        block.norm1.scale = nn.Parameter(torch.tensor(params["blocks"][b]["ln_1"]["g"]))
        block.norm1.shift = nn.Parameter(torch.tensor(params["blocks"][b]["ln_1"]["b"]))
        block.norm2.scale = nn.Parameter(torch.tensor(params["blocks"][b]["ln_2"]["g"]))
        block.norm2.shift = nn.Parameter(torch.tensor(params["blocks"][b]["ln_2"]["b"]))
    gpt.final_norm.scale = nn.Parameter(torch.tensor(params["g"]))
    gpt.final_norm.shift = nn.Parameter(torch.tensor(params["b"]))
    gpt.out_head.weight  = nn.Parameter(torch.tensor(params["wte"]))  # tied to token embedding

# GPT-2 Medium config (dropout off + qkv bias on, as OpenAI's weights expect)
INS_CONFIG = {
    "vocab_size": 50257, "context_length": 1024,
    "emb_dim": 1024, "n_heads": 16, "n_layers": 24,   # Medium
    "drop_rate": 0.0, "qkv_bias": True
}
# NOTE: if the T4 runs out of memory, switch to the 124M model instead:
#   INS_CONFIG = {**GPT_CONFIG_TRAIN, "context_length":1024, "drop_rate":0.0, "qkv_bias":True}
#   download_and_load_gpt2(model_size="124M", ...)

settings_m, params_m = download_and_load_gpt2(model_size="355M", models_dir="gpt2")
ins_model = GPTModel(INS_CONFIG)
load_weights_into_gpt(ins_model, params_m)
ins_model.eval()
print("GPT-2 Medium (355M) loaded ✓")

In [ ]:
# ── Instruction-tuning helpers (define once; used by every fine-tune cell) ────
# Split into its own cell so you can set up the helpers WITHOUT triggering a full
# training run -- handy for the Exercise-3 lean path where you only want to train
# the congress model.
def calc_loss_loader_ins(loader, model, device, num_batches=None):
    """Average cross-entropy over a few batches; ignore_index=-100 skips masked tokens."""
    total, n = 0., min(num_batches or len(loader), len(loader))
    for i, (xb, yb) in enumerate(loader):
        if i >= n: break
        xb, yb = xb.to(device), yb.to(device)
        total += torch.nn.functional.cross_entropy(
            model(xb).flatten(0, 1), yb.flatten(), ignore_index=-100
        ).item()
    return total / n

def train_instruction_model(model, train_loader, val_loader, optimizer,
                            device, num_epochs, eval_freq, eval_iter,
                            val_sample, tokenizer):
    """Standard instruction fine-tuning loop; prints a sample answer per epoch."""
    train_losses, val_losses, tokens_seen = [], [], []
    seen, step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            xb, yb = xb.to(device), yb.to(device)
            loss = torch.nn.functional.cross_entropy(
                model(xb).flatten(0, 1), yb.flatten(), ignore_index=-100
            )
            loss.backward(); optimizer.step()
            seen += xb.numel(); step += 1
            if step % eval_freq == 0:
                tl = calc_loss_loader_ins(train_loader, model, device, eval_iter)
                vl = calc_loss_loader_ins(val_loader,   model, device, eval_iter)
                train_losses.append(tl); val_losses.append(vl); tokens_seen.append(seen)
                print(f"Ep {epoch+1} Step {step:04d}: train={tl:.3f}  val={vl:.3f}")
        # one sample per epoch. Feed the prompt INCLUDING "### Response:\n" so a
        # masked model (which never learned to emit the header) still answers.
        model.eval()
        prompt = format_input(val_sample) + "\n\n### Response:\n"
        token_ids = generate(model, text_to_token_ids(prompt, tokenizer).to(device),
                             max_new_tokens=80, context_size=INS_CONFIG["context_length"],
                             temperature=0.0, eos_id=50256)
        resp = token_ids_to_text(token_ids, tokenizer)[len(prompt):].strip()
        print(f"  Sample response: {resp[:120]}...")
        model.train()
    return train_losses, val_losses, tokens_seen

print("Instruction-tuning helpers defined ✓")

In [ ]:
# ── Build instruction-MASKED DataLoaders and fine-tune ───────────────────────
# Reuses InstructionDatasetWithMask + the masked custom_collate_fn from Exercise 2,
# and train_instruction_model / calc_loss_loader_ins from the helpers cell above.
customized_collate = partial(custom_collate_fn, device=device, allowed_max_length=1024)

train_ins_ds = InstructionDatasetWithMask(train_data, tokenizer)
val_ins_ds   = InstructionDatasetWithMask(val_data,   tokenizer)

train_ins_loader = DataLoader(train_ins_ds, batch_size=8, shuffle=True,
                               drop_last=True,  collate_fn=customized_collate)
val_ins_loader   = DataLoader(val_ins_ds,   batch_size=8, shuffle=False,
                               drop_last=False, collate_fn=customized_collate)

# Sanity-check the starting loss, then train (~a few minutes on the T4)
ins_model.to(device)
with torch.no_grad():
    print(f"Initial — train={calc_loss_loader_ins(train_ins_loader, ins_model, device, 5):.3f}  "
          f"val={calc_loss_loader_ins(val_ins_loader, ins_model, device, 5):.3f}")

torch.manual_seed(123)
optimizer_ins = torch.optim.AdamW(ins_model.parameters(), lr=5e-5, weight_decay=0.1)

t0 = time.time()
train_losses_ins, val_losses_ins, tokens_seen_ins = train_instruction_model(
    ins_model, train_ins_loader, val_ins_loader, optimizer_ins, device,
    num_epochs=2, eval_freq=5, eval_iter=5,
    val_sample=val_data[0], tokenizer=tokenizer
)
print(f"\nTraining completed in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── Loss curves for the masked fine-tuning run ───────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(train_losses_ins, label="Train loss")
ax.plot(val_losses_ins,   label="Val loss", linestyle="-.")
ax.set_xlabel("Evaluation step"); ax.set_ylabel("Loss")
ax.set_title("Instruction Fine-tuning Loss (instruction-masked)")
ax.legend(); plt.tight_layout(); plt.show()

# ── Inspect a few test responses from the masked model ───────────────────────
# IMPORTANT: instruction masking also masks the "### Response:\n" header, so the
# masked model was never trained to GENERATE that header on its own -- it only
# learned to produce the answer *after* it. So at inference we must feed the
# prompt INCLUDING "\n\n### Response:\n" and let the model continue with just the
# answer. (Feeding the header explicitly is the standard Alpaca convention and
# works for the unmasked baseline too.) Without this, the masked model emits EOS
# immediately and you get empty responses.
ins_model.to(device).eval()   # .to(device) in case the baseline cell moved it to CPU
for entry in test_data[:3]:
    prompt = format_input(entry) + "\n\n### Response:\n"
    token_ids = generate(
        ins_model, text_to_token_ids(prompt, tokenizer).to(device),
        max_new_tokens=100, context_size=INS_CONFIG["context_length"],
        temperature=0.0, eos_id=50256
    )
    response = token_ids_to_text(token_ids, tokenizer)[len(prompt):].strip()
    print(f"Instruction: {entry['instruction']}")
    if entry["input"]: print(f"Input:       {entry['input']}")
    print(f"Expected:    {entry['output']}")
    print(f"Model:       {response[:200]}")
    print("-" * 60)

### Compare against an unmasked baseline

To judge whether instruction masking helped, train a second model with the **exact same** recipe but using the original (padding-only) collate fn, then compare the two on a **response-only** validation loss and on sample outputs.

> ⚠️ This trains a *second* GPT-2 Medium. To stay within T4 memory it first moves the masked model to CPU and frees its optimizer — it does **not** delete it, so the masked cells above remain re-runnable (re-running the masked-sample cell moves it back to the GPU). Run the masked training + its plot/sample cell before this one.

In [ ]:
# ── Baseline: identical setup, but WITHOUT instruction masking ───────────────
# Free GPU memory from the masked run, but KEEP the masked model so the cells
# above stay re-runnable. We move it to CPU (frees ~1.4 GB of GPU) and drop its
# optimizer (the Adam state is the real GPU hog, ~2.8 GB). If you re-run the
# masked-sample cell later it will move ins_model back onto the GPU automatically.
import gc
ins_model.to("cpu")
del optimizer_ins
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

# Unmasked dataset: stores just the encoded full text (instruction + response)
class InstructionDatasetBaseline(Dataset):
    def __init__(self, data, tokenizer):
        self.encoded = [
            tokenizer.encode(format_input(e) + "\n\n### Response:\n" + e["output"])
            for e in data
        ]
    def __len__(self): return len(self.encoded)
    def __getitem__(self, idx): return self.encoded[idx]

# Original lab collate: masks ONLY padding (keeps one EOS), no instruction masking
def custom_collate_fn_baseline(batch, pad_token_id=50256, ignore_index=-100,
                               allowed_max_length=None, device="cpu"):
    max_len = max(len(ids) + 1 for ids in batch)
    inputs_lst, targets_lst = [], []
    for ids in batch:
        padded  = ids + [pad_token_id] * (max_len - len(ids))
        inputs  = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        pad_positions = (targets == pad_token_id).nonzero().squeeze(-1)
        if pad_positions.numel() > 1:
            targets[pad_positions[1:]] = ignore_index
        if allowed_max_length:
            inputs  = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]
        inputs_lst.append(inputs); targets_lst.append(targets)
    return torch.stack(inputs_lst).to(device), torch.stack(targets_lst).to(device)

base_collate = partial(custom_collate_fn_baseline, device=device, allowed_max_length=1024)
train_base_loader = DataLoader(InstructionDatasetBaseline(train_data, tokenizer),
                               batch_size=8, shuffle=True,  drop_last=True,  collate_fn=base_collate)
val_base_loader   = DataLoader(InstructionDatasetBaseline(val_data,   tokenizer),
                               batch_size=8, shuffle=False, drop_last=False, collate_fn=base_collate)

# Fresh pretrained model + identical training recipe (same seed, lr, epochs)
base_model = GPTModel(INS_CONFIG)
load_weights_into_gpt(base_model, params_m)
base_model.to(device)

torch.manual_seed(123)
optimizer_base = torch.optim.AdamW(base_model.parameters(), lr=5e-5, weight_decay=0.1)

t0 = time.time()
train_losses_base, val_losses_base, _ = train_instruction_model(
    base_model, train_base_loader, val_base_loader, optimizer_base, device,
    num_epochs=2, eval_freq=5, eval_iter=5,
    val_sample=val_data[0], tokenizer=tokenizer
)
print(f"\nUnmasked baseline training completed in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── Side-by-side comparison ──────────────────────────────────────────────────
# Fair, apples-to-apples metric: RESPONSE-ONLY validation loss for both models.
# val_ins_loader is the masked loader, so loss over it ignores instruction tokens.
# The masked model's response-only loss is just its final training val loss.
masked_resp_val = val_losses_ins[-1]
base_resp_val   = calc_loss_loader_ins(val_ins_loader, base_model, device)
print(f"Response-only val loss  |  masked: {masked_resp_val:.3f}   unmasked: {base_resp_val:.3f}")
print("(lower = better; this is the fair comparison since both are scored on response tokens only)\n")

# Loss curves -- note each curve is plotted against its OWN training objective
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(val_losses_ins,  label="masked (response-only loss)")
ax.plot(val_losses_base, label="unmasked (all-token loss)", linestyle="-.")
ax.set_xlabel("Evaluation step"); ax.set_ylabel("Val loss")
ax.set_title("Masked vs. unmasked instruction fine-tuning")
ax.legend(); plt.tight_layout(); plt.show()

# Sample responses from the UNMASKED baseline (compare to the masked ones above).
# Use the SAME header-inclusive prompt as the masked model so the two are decoded
# identically (the baseline also learned to emit the header, so either works).
base_model.eval()
for entry in test_data[:3]:
    prompt = format_input(entry) + "\n\n### Response:\n"
    ids = generate(base_model, text_to_token_ids(prompt, tokenizer).to(device),
                   max_new_tokens=100, context_size=INS_CONFIG["context_length"],
                   temperature=0.0, eos_id=50256)
    resp = token_ids_to_text(ids, tokenizer)[len(prompt):].strip()
    print(f"Instruction: {entry['instruction']}")
    if entry["input"]: print(f"Input:       {entry['input']}")
    print(f"Expected:    {entry['output']}")
    print(f"Unmasked:    {resp[:200]}")
    print("-" * 60)

### Masked vs. unmasked — what's actually different?

Both runs use the **same** pretrained model, data, hyperparameters, and number of steps. The *only* difference is **which target tokens contribute to the loss**:

| | Tokens that produce a loss/gradient | What the model is asked to learn |
|---|---|---|
| **Unmasked (baseline)** | every token: the prompt template, the instruction, the input, **and** the response | model the *whole* sequence — re-predict the boilerplate prompt as well as the answer |
| **Masked** | **only the response tokens** (instruction tokens set to `-100`) | model *only* the answer, given the prompt as fixed context |

Mechanically, masking sets `targets[:instruction_len - 1] = -100`, and `cross_entropy(..., ignore_index=-100)` drops those positions from both the loss and the backward pass.

**A practical consequence at inference (we hit this above):** the `"\n\n### Response:\n"` header is part of the *instruction*, so masking it means the masked model never gets gradient to **generate** that header — it only learns to produce the answer *after* it. If you prompt it with `format_input(entry)` alone (expecting it to emit the header like the unmasked model does), it instead emits EOS and you get an **empty response**. Fix: feed the prompt *including* `"\n\n### Response:\n"` so it continues with the answer. (We now do this for both models so they're decoded identically.)

**Why the loss numbers differ (and aren't directly comparable):** the baseline loss averages over *all* tokens (including the easy, repetitive template, which it learns to predict very confidently → lower average), while the masked loss averages over *only* response tokens (the genuinely hard part → higher average). That's why we report a **response-only val loss for both models** above: evaluating the baseline through the masked val loader puts them on the same footing.

**What to expect here:** with a 355M model and ~1k examples, the two usually end up close, with the **unmasked baseline often slightly ahead** on response quality — the extra next-token signal from the instruction tokens helps more than the "focus" from masking does at this scale. Masking tends to win when data is plentiful and you don't want to waste capacity memorizing prompt boilerplate. Compare the **response-only val loss** and the **sample answers** above to see which won on your run.

**Exercise 3 — Applying instruction fine-tuning to congressional speeches**

We have a dataset of 28,000+ congressional floor speeches with party and date metadata. Design an instruction dataset from this corpus. For example:

- Instruction: `"Summarize this Democratic speech in one sentence."`
- Input: `[speech text]`
- Output: `[your summary]`

Or more interestingly:

- Instruction: `"What policy does this speech advocate for?"`
- Input: `[speech text]`
- Output: `[policy label]`

Sketch (in markdown) how you would construct `instruction-data.json` entries from the `us_congress_speeches_sample.csv` file. What would you use as the `output` field? What are the tradeoffs of different designs?

**Design sketch**

The CSV has columns `date, text, speaker_bioguide, chamber, party, doc_clean`. The decisive fact is that **some fields are ground-truth labels (`party`, `chamber`, `date`) while a "summary" or "policy" field does not exist.** That splits the designs into two families:

**1. Label-grounded tasks (recommended — output already exists)**
Map an existing column to the `output` field, the speech to `input`, and a fixed question to `instruction`:

| Task | `instruction` | `input` | `output` |
|---|---|---|---|
| Party classification | "Identify the speaker's party. Answer Democrat or Republican." | speech `text` | `party` |
| Chamber classification | "Was this delivered in the House or the Senate?" | speech `text` | `chamber` |
| Decade / era | "Roughly what decade is this speech from?" | speech `text` | decade from `date` |

These are cheap, reproducible, and have **no label noise** — the output is a real attribute of the record. The cost is that they're discriminative tasks dressed up as instructions; the model mostly learns a classifier, and the `output` is a single token, so there isn't much *generative* skill being taught.

**2. Generative tasks (summary / policy stance — output must be created)**
There is no human reference, so you must **manufacture** the `output`:
- **LLM distillation:** prompt a stronger model (e.g. Claude/GPT) to summarize or extract the policy stance, and use that as `output`. Highest quality, but costs API calls and inherits the teacher's biases/errors.
- **Weak/heuristic labels:** use `doc_clean` (already a keyword-reduced version) or the first 1–2 sentences as a pseudo-summary; or derive a topic label from a topic model. Free, but noisy and not really "summaries."
- **Self-instruct:** have the model you're training propose outputs, then filter — circular and risky at this scale.

**What I'd use for `output`:** start with **`party`** (Design A in the code below) because it's a clean, balanced, verifiable label that exercises the full instruction→input→response format end-to-end. If a generative task is required, I'd use **LLM-distilled one-sentence summaries** and treat them as silver labels.

**Trade-offs to weigh**
- **Label reliability vs. task richness:** metadata labels are reliable but shallow; generated summaries are richer but noisy/expensive.
- **Class balance & shortcuts:** parties are imbalanced over time and a model can latch onto era cues (e.g. names, bill numbers) rather than ideology — balance classes and consider stratifying by `date`.
- **Context length:** speeches can exceed the 1024-token window, so `input` must be clipped/truncated (done in the code), which can drop the very content that carries the signal.
- **Leakage & ethics:** predicting party from speech is a partisanship classifier — fine for a class exercise, but worth flagging that such a model encodes political bias and shouldn't be deployed naively.

The cell below implements the recommended **party-classification** design and writes a balanced `instruction-data-congress.json` in the exact `{instruction, input, output}` schema the Exercise-2 pipeline expects.

In [ ]:
# ── Exercise 3: build an instruction dataset from the congressional speeches ──
# The CSV columns are: date, text, speaker_bioguide, chamber, party, doc_clean.
# Key idea: party/chamber are REAL labels already in the data, so we can build a
# reliable instruction dataset for free. (Summaries would have no ground truth --
# see the markdown sketch above for that trade-off.)
import pandas as pd, json, os

CSV = "us_congress_speeches_sample.csv"
# On Colab: upload this course file via the Files panel (or mount Google Drive).
assert os.path.exists(CSV), f"Upload {CSV} to the Colab session first (Files -> Upload)."

# Read only the columns we need
df = pd.read_csv(CSV, usecols=["date", "text", "party", "chamber"])
df = df.dropna(subset=["text", "party"])
df["text"] = df["text"].astype(str).str.strip()
df = df[df["party"].isin(["Democrat", "Republican"])]      # focus on the two major parties
df = df[df["text"].str.len().between(200, 2000)]           # drop fragments / over-long speeches

# Keep prompts within the model's context window by clipping the speech text
def clip(t, n=1200):
    return (t[:n].rsplit(" ", 1)[0] + " ...") if len(t) > n else t

# Design A — PARTY classification (the label exists, so output is reliable)
def make_party_entry(row):
    return {
        "instruction": ("Identify the political party of the member of Congress who gave this "
                        "floor speech. Answer with a single word: Democrat or Republican."),
        "input":  clip(row["text"]),
        "output": row["party"],
    }

# Balance the two classes so the model can't win by always predicting the majority label
n_per = 1500
balanced = pd.concat([
    df[df.party == "Democrat"].sample(n_per, random_state=0),
    df[df.party == "Republican"].sample(n_per, random_state=0),
]).sample(frac=1, random_state=0)   # shuffle

entries = [make_party_entry(r) for _, r in balanced.iterrows()]

with open("instruction-data-congress.json", "w") as f:
    json.dump(entries, f, indent=2)

print(f"Built {len(entries)} entries  ->  instruction-data-congress.json")
print("\nExample entry:")
print(json.dumps(entries[0], indent=2)[:700])

# To fine-tune on this, reuse the Exercise-2 pipeline:
#   data = entries
#   train_data, test_data, val_data = <re-split as before>
#   train_ds = InstructionDatasetWithMask(train_data, tokenizer)   # or the baseline dataset
#   ... then DataLoaders + train_instruction_model(...) exactly as above.

### Fine-tune on the congressional dataset

Now actually instruction-fine-tune GPT-2 Medium on the party-classification data built above, reusing the same masked pipeline (`InstructionDatasetWithMask` + `custom_collate_fn` + `train_instruction_model`). Then evaluate **test accuracy** — since this is a balanced binary classification task, accuracy is more meaningful than loss.

**Run order:** the GPT-2 Medium loader cell and the Exercise-3 builder cell must have run first (they define `params_m`/`load_weights_into_gpt`/`INS_CONFIG` and `entries`). This loads a *fresh* model, so it's independent of the masked/unmasked runs above.

In [ ]:
# ── Fine-tune GPT-2 Medium on the congressional party-classification task ─────
# LEAN PATH (recommended): after a runtime restart, run ONLY these cells so just
# ONE 355M model is ever on the GPU:
#   SETUP 1/3 + SETUP 2/3 (classes, generate, tokenizer, device)
#   Exercise-2 solution cell      -> custom_collate_fn, InstructionDatasetWithMask, format_input, data split
#   Instruction-tuning helpers    -> train_instruction_model, calc_loss_loader_ins
#   GPT-2 Medium loader cell      -> INS_CONFIG, params_m, load_weights_into_gpt
#   Exercise-3 builder cell       -> entries
# You do NOT need to run the alpaca masked/baseline training cells for this.
import gc, random
from functools import partial

_missing = [name for name in
            ["custom_collate_fn", "InstructionDatasetWithMask", "train_instruction_model",
             "INS_CONFIG", "params_m", "load_weights_into_gpt", "entries"]
            if name not in globals()]
assert not _missing, f"Run the prereq cells first — missing: {_missing}"

# Re-split the congress entries into train / test / val (85 / 10 / 5)
random.seed(0)
random.shuffle(entries)
n = len(entries)
n_tr, n_te = int(0.85 * n), int(0.10 * n)
congress_train = entries[:n_tr]
congress_test  = entries[n_tr:n_tr + n_te]
congress_val   = entries[n_tr + n_te:]
print(f"train={len(congress_train)}  val={len(congress_val)}  test={len(congress_test)}")

# ── Free GPU memory from any earlier runs ────────────────────────────────────
# Deleting a model does NOT free its optimizer's Adam state, so delete BOTH.
# (If you took the lean path above there's nothing to free; this is just a safety net.)
for _name in ["ins_model", "base_model", "optimizer_ins", "optimizer_base",
              "congress_model", "opt_c"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"GPU free: {free_b/1e9:.1f} / {total_b/1e9:.1f} GB after cleanup")

# ── Masked datasets/loaders ──────────────────────────────────────────────────
# Single-word label -> mask the long speech prompt, train only on the label.
# Attention memory is O(n^2) x batch and speeches are long, so use a small batch.
# allowed_max_length=512 caps the sequence; clipped speeches stay under it so the
# label at the end is never truncated.
BATCH = 4   # lower to 2 or 1 if you OOM; raise if you have headroom
c_collate = partial(custom_collate_fn, device=device, allowed_max_length=512)
c_train_loader = DataLoader(InstructionDatasetWithMask(congress_train, tokenizer),
                            batch_size=BATCH, shuffle=True,  drop_last=True,  collate_fn=c_collate)
c_val_loader   = DataLoader(InstructionDatasetWithMask(congress_val,   tokenizer),
                            batch_size=BATCH, shuffle=False, drop_last=False, collate_fn=c_collate)

# Fresh pretrained GPT-2 Medium (don't continue training an already-tuned model)
congress_model = GPTModel(INS_CONFIG)
load_weights_into_gpt(congress_model, params_m)
congress_model.to(device)

torch.manual_seed(123)
opt_c = torch.optim.AdamW(congress_model.parameters(), lr=5e-5, weight_decay=0.1)

t0 = time.time()
c_train_losses, c_val_losses, _ = train_instruction_model(
    congress_model, c_train_loader, c_val_loader, opt_c, device,
    num_epochs=2, eval_freq=50, eval_iter=5,
    val_sample=congress_val[0], tokenizer=tokenizer
)
print(f"\nCongress fine-tune completed in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── Evaluate: exact-match accuracy on the held-out test set ───────────────────
# This is a classification task, so accuracy (not loss) is the natural metric.
# Chance is ~50% because the data is class-balanced.
congress_model.eval()
N = min(200, len(congress_test))   # subset for speed
correct, shown = 0, 0
for entry in congress_test[:N]:
    prompt = format_input(entry) + "\n\n### Response:\n"
    ids = generate(congress_model, text_to_token_ids(prompt, tokenizer).to(device),
                   max_new_tokens=5, context_size=INS_CONFIG["context_length"],
                   temperature=0.0, eos_id=50256)
    gen = token_ids_to_text(ids, tokenizer)[len(prompt):].strip()
    pred = gen.split()[0].strip(".,!").capitalize() if gen.split() else ""
    correct += int(pred == entry["output"])
    if shown < 6:
        flag = "✓" if pred == entry["output"] else "✗"
        print(f"{flag} pred={pred or '(empty)':11s} gold={entry['output']}")
        shown += 1

print(f"\nTest accuracy ({N} examples): {correct / N:.1%}   (50% = chance on balanced classes)")

### ⭐ Exercise 3 — standalone, single-cell version

Self-contained: **Restart session**, upload `us_congress_speeches_sample.csv` (Files panel), then run the one cell below. It does everything (imports, model, dataset, fine-tune, accuracy) on a fresh GPU.

In [ ]:
# =============================================================================
# EXERCISE 3 — STANDALONE.  Restart the runtime, upload the CSV, run ONLY this cell.
# =============================================================================

# (0) Reduce CUDA fragmentation. MUST be set before torch initializes CUDA.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# (1) Installs (tiktoken for the tokenizer; tensorflow is used by the GPT-2 downloader)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken"], check=True)

# (2) Imports
import json, time, random, gc, urllib.request
import numpy as np
import pandas as pd
import torch, torch.nn as nn, tiktoken
from torch.utils.data import Dataset, DataLoader
from functools import partial

tokenizer = tiktoken.get_encoding("gpt2")
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# (3) GPT-2 model definition (from the lab) ------------------------------------
class LayerNorm(nn.Module):
    def __init__(self, d):
        super().__init__(); self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(d)); self.shift = nn.Parameter(torch.zeros(d))
    def forward(self, x):
        m = x.mean(-1, keepdim=True); v = x.var(-1, keepdim=True, unbiased=False)
        return self.scale * (x - m) / torch.sqrt(v + self.eps) + self.shift

class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi)) * (x + 0.044715*x**3)))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]), GELU(),
                                    nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"]))
    def forward(self, x): return self.layers(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, ctx, dropout, n_heads, qkv_bias=False):
        super().__init__()
        self.n_heads = n_heads; self.head_dim = d_out // n_heads; self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out); self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(ctx, ctx), diagonal=1))
    def forward(self, x):
        b, n, _ = x.shape
        q = self.W_query(x).view(b, n, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.W_key(x).view(b, n, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.W_value(x).view(b, n, self.n_heads, self.head_dim).transpose(1, 2)
        scores = q @ k.transpose(2, 3)
        scores.masked_fill_(self.mask.bool()[:n, :n], -torch.inf)
        w = self.dropout(torch.softmax(scores / k.shape[-1]**0.5, dim=-1))
        return self.out_proj((w @ v).transpose(1, 2).contiguous().view(b, n, self.d_out))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"], cfg["context_length"],
                                      cfg["drop_rate"], cfg["n_heads"], cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"]); self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop = nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        x = x + self.drop(self.att(self.norm1(x)))
        x = x + self.drop(self.ff(self.norm2(x)))
        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
    def forward(self, idx):
        b, n = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(n, device=idx.device))
        return self.out_head(self.final_norm(self.trf_blocks(self.drop_emb(x))))

def text_to_token_ids(t, tok): return torch.tensor(tok.encode(t, allowed_special={"<|endoftext|>"})).unsqueeze(0)
def token_ids_to_text(ids, tok): return tok.decode(ids.squeeze(0).tolist())

def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(idx[:, -context_size:])[:, -1, :]
        if top_k is not None:
            top, _ = torch.topk(logits, top_k)
            logits = torch.where(logits < top[:, -1], torch.tensor(float("-inf")).to(logits.device), logits)
        if temperature > 0.0:
            idx_next = torch.multinomial(torch.softmax(logits/temperature, -1), 1)
        else:
            idx_next = torch.argmax(logits, -1, keepdim=True)
        if eos_id is not None and idx_next.item() == eos_id: break
        idx = torch.cat((idx, idx_next), 1)
    return idx

# (4) Load pretrained GPT-2 Medium (355M) -------------------------------------
if not os.path.exists("gpt_download.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/"
        "main/ch05/01_main-chapter-code/gpt_download.py", "gpt_download.py")
from gpt_download import download_and_load_gpt2

def load_weights_into_gpt(gpt, p):
    gpt.pos_emb.weight = nn.Parameter(torch.tensor(p["wpe"]))
    gpt.tok_emb.weight = nn.Parameter(torch.tensor(p["wte"]))
    for b, blk in enumerate(gpt.trf_blocks):
        q, k, v = np.split(p["blocks"][b]["attn"]["c_attn"]["w"], 3, -1)
        blk.att.W_query.weight = nn.Parameter(torch.tensor(q).T)
        blk.att.W_key.weight   = nn.Parameter(torch.tensor(k).T)
        blk.att.W_value.weight = nn.Parameter(torch.tensor(v).T)
        qb, kb, vb = np.split(p["blocks"][b]["attn"]["c_attn"]["b"], 3)
        blk.att.W_query.bias = nn.Parameter(torch.tensor(qb))
        blk.att.W_key.bias   = nn.Parameter(torch.tensor(kb))
        blk.att.W_value.bias = nn.Parameter(torch.tensor(vb))
        blk.att.out_proj.weight = nn.Parameter(torch.tensor(p["blocks"][b]["attn"]["c_proj"]["w"]).T)
        blk.att.out_proj.bias   = nn.Parameter(torch.tensor(p["blocks"][b]["attn"]["c_proj"]["b"]))
        blk.ff.layers[0].weight = nn.Parameter(torch.tensor(p["blocks"][b]["mlp"]["c_fc"]["w"]).T)
        blk.ff.layers[0].bias   = nn.Parameter(torch.tensor(p["blocks"][b]["mlp"]["c_fc"]["b"]))
        blk.ff.layers[2].weight = nn.Parameter(torch.tensor(p["blocks"][b]["mlp"]["c_proj"]["w"]).T)
        blk.ff.layers[2].bias   = nn.Parameter(torch.tensor(p["blocks"][b]["mlp"]["c_proj"]["b"]))
        blk.norm1.scale = nn.Parameter(torch.tensor(p["blocks"][b]["ln_1"]["g"]))
        blk.norm1.shift = nn.Parameter(torch.tensor(p["blocks"][b]["ln_1"]["b"]))
        blk.norm2.scale = nn.Parameter(torch.tensor(p["blocks"][b]["ln_2"]["g"]))
        blk.norm2.shift = nn.Parameter(torch.tensor(p["blocks"][b]["ln_2"]["b"]))
    gpt.final_norm.scale = nn.Parameter(torch.tensor(p["g"]))
    gpt.final_norm.shift = nn.Parameter(torch.tensor(p["b"]))
    gpt.out_head.weight  = nn.Parameter(torch.tensor(p["wte"]))

INS_CONFIG = {"vocab_size": 50257, "context_length": 1024, "emb_dim": 1024,
              "n_heads": 16, "n_layers": 24, "drop_rate": 0.0, "qkv_bias": True}
_, params_m = download_and_load_gpt2(model_size="355M", models_dir="gpt2")
print("GPT-2 Medium loaded ✓")

# (5) Instruction-masking dataset + collate + training loop -------------------
def format_input(e):
    txt = ("Below is an instruction that describes a task. Write a response that "
           "appropriately completes the request.\n\n### Instruction:\n" + e["instruction"])
    return txt + (f"\n\n### Input:\n{e['input']}" if e["input"] else "")

class InstructionDatasetWithMask(Dataset):
    def __init__(self, data, tok):
        self.items = []
        for e in data:
            head = format_input(e) + "\n\n### Response:\n"
            self.items.append((tok.encode(head + e["output"]), len(tok.encode(head))))
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def custom_collate_fn(batch, pad=50256, ignore=-100, max_len_cap=None, device="cpu"):
    L = max(len(ids) + 1 for ids, _ in batch)
    X, Y = [], []
    for ids, inst_len in batch:
        padded = ids + [pad] * (L - len(ids))
        x = torch.tensor(padded[:-1]); y = torch.tensor(padded[1:])
        pads = (y == pad).nonzero().squeeze(-1)
        if pads.numel() > 1: y[pads[1:]] = ignore     # keep one EOS, mask extra padding
        y[:inst_len - 1] = ignore                      # mask the instruction tokens
        if max_len_cap: x, y = x[:max_len_cap], y[:max_len_cap]
        X.append(x); Y.append(y)
    return torch.stack(X).to(device), torch.stack(Y).to(device)

def calc_loss(loader, model, device, nb=None):
    tot, n = 0., min(nb or len(loader), len(loader))
    for i, (xb, yb) in enumerate(loader):
        if i >= n: break
        xb, yb = xb.to(device), yb.to(device)
        tot += torch.nn.functional.cross_entropy(model(xb).flatten(0,1), yb.flatten(), ignore_index=-100).item()
    return tot / n

def train_model(model, tr, va, opt, device, epochs, eval_freq, eval_iter, val_sample):
    trL, vaL = [], []; step = -1
    for ep in range(epochs):
        model.train()
        for xb, yb in tr:
            opt.zero_grad(); xb, yb = xb.to(device), yb.to(device)
            loss = torch.nn.functional.cross_entropy(model(xb).flatten(0,1), yb.flatten(), ignore_index=-100)
            loss.backward(); opt.step(); step += 1
            if step % eval_freq == 0:
                tl, vl = calc_loss(tr, model, device, eval_iter), calc_loss(va, model, device, eval_iter)
                trL.append(tl); vaL.append(vl); print(f"Ep {ep+1} Step {step:04d}: train={tl:.3f} val={vl:.3f}")
        model.eval()
        prompt = format_input(val_sample) + "\n\n### Response:\n"
        ids = generate(model, text_to_token_ids(prompt, tokenizer).to(device), 10,
                       INS_CONFIG["context_length"], temperature=0.0, eos_id=50256)
        print("  sample:", token_ids_to_text(ids, tokenizer)[len(prompt):].strip()[:60])
        model.train()
    return trL, vaL

# (6) Build the congressional party-classification dataset from the CSV --------
CSV = "us_congress_speeches_sample.csv"
assert os.path.exists(CSV), f"Upload {CSV} to the Colab session first (Files -> Upload)."
df = pd.read_csv(CSV, usecols=["text", "party"]).dropna(subset=["text", "party"])
df["text"] = df["text"].astype(str).str.strip()
df = df[df["party"].isin(["Democrat", "Republican"])]
df = df[df["text"].str.len().between(200, 2000)]

def clip(t, n=1200): return (t[:n].rsplit(" ", 1)[0] + " ...") if len(t) > n else t
def make_entry(row):
    return {"instruction": ("Identify the political party of the member of Congress who gave this "
                            "floor speech. Answer with a single word: Democrat or Republican."),
            "input": clip(row["text"]), "output": row["party"]}

n_per = 1500
bal = pd.concat([df[df.party == "Democrat"].sample(n_per, random_state=0),
                 df[df.party == "Republican"].sample(n_per, random_state=0)]).sample(frac=1, random_state=0)
entries = [make_entry(r) for _, r in bal.iterrows()]
random.seed(0); random.shuffle(entries)
n = len(entries); n_tr, n_te = int(0.85*n), int(0.10*n)
train_data, test_data, val_data = entries[:n_tr], entries[n_tr:n_tr+n_te], entries[n_tr+n_te:]
print(f"dataset: train={len(train_data)} val={len(val_data)} test={len(test_data)}")

# (7) Fine-tune ----------------------------------------------------------------
BATCH = 4   # lower to 2/1 if you OOM
collate = partial(custom_collate_fn, device=device, max_len_cap=512)
tr_loader = DataLoader(InstructionDatasetWithMask(train_data, tokenizer), batch_size=BATCH,
                       shuffle=True, drop_last=True, collate_fn=collate)
va_loader = DataLoader(InstructionDatasetWithMask(val_data, tokenizer), batch_size=BATCH,
                       shuffle=False, drop_last=False, collate_fn=collate)

model = GPTModel(INS_CONFIG); load_weights_into_gpt(model, params_m); model.to(device)
torch.manual_seed(123)
opt = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
t0 = time.time()
train_model(model, tr_loader, va_loader, opt, device, epochs=2, eval_freq=50, eval_iter=5, val_sample=val_data[0])
print(f"fine-tune done in {(time.time()-t0)/60:.1f} min")

# (8) Test accuracy ------------------------------------------------------------
model.eval(); N = min(200, len(test_data)); correct = 0
for i, e in enumerate(test_data[:N]):
    prompt = format_input(e) + "\n\n### Response:\n"
    ids = generate(model, text_to_token_ids(prompt, tokenizer).to(device), 5,
                   INS_CONFIG["context_length"], temperature=0.0, eos_id=50256)
    gen = token_ids_to_text(ids, tokenizer)[len(prompt):].strip()
    pred = gen.split()[0].strip(".,!").capitalize() if gen.split() else ""
    correct += int(pred == e["output"])
    if i < 6:
        print(("✓" if pred == e["output"] else "✗"), f"pred={pred or '(empty)':11s} gold={e['output']}")
print(f"\nTest accuracy ({N} examples): {correct/N:.1%}   (50% = chance on balanced classes)")

In [ ]:
# ── Show example speeches + the model's predictions ──────────────────────────
# Reuses the already-trained `model` and `test_data` from the cell above
# (no retraining needed). Prints the speech text, the true party, and the prediction.
model.eval()
for e in test_data[:10]:
    prompt = format_input(e) + "\n\n### Response:\n"
    ids = generate(model, text_to_token_ids(prompt, tokenizer).to(device), 5,
                   INS_CONFIG["context_length"], temperature=0.0, eos_id=50256)
    gen  = token_ids_to_text(ids, tokenizer)[len(prompt):].strip()
    pred = gen.split()[0].strip(".,!").capitalize() if gen.split() else "(empty)"
    flag = "✓" if pred == e["output"] else "✗"
    print(f"{flag}  predicted: {pred:11s} |  actual: {e['output']}")
    print(f"    speech: {e['input'][:240].strip()} ...")
    print("-" * 95)